# Anomaly detection on sensor time series

Five detectors on nine labeled series from the [Numenta Anomaly Benchmark](https://github.com/numenta/NAB):
rolling z-score (baseline), Isolation Forest on window features, matrix profile discords (stumpy, z-normalized and raw)
and a 1D-conv autoencoder.
The sixth entry, `ensemble`, averages the validation-calibrated scores of the raw matrix profile and the z-score at twice the context length; the leave-one-out optimization at the end selected this combination.

Each series is split chronologically into train (40%), validation (20%) and test (40%). Detectors are fit on train with labeled
windows removed. For series with a weekly cycle (nyc_taxi, ambient_temperature) the median time-of-week profile, estimated on
normal training points, is subtracted first. The threshold is the same score quantile on normal validation points for every detector.

Evaluation runs on test at event level: a labeled window counts as detected if at least one alarm falls inside it, each alarm run
outside a window is one false positive. Alarms within one context length after a window are ignored, because the trailing window
still contains the anomaly. Isolation Forest and autoencoder are run with five seeds.

The second part repeats the comparison on the multivariate [SKAB](https://github.com/waico/SKAB) benchmark, explains which sensor
drives each alarm and shows the detector running point by point.

Interactive version with a threshold slider: [Streamlit app](https://anomalydetectionsensor-53njzwcxxrjku7qqqrjpns.streamlit.app/).

In [1]:
import json

import numpy as np
import pandas as pd
import plotly.io as pio

from src.data import ROOT
from src.plots import attribution_heatmap, hero, small_multiples, sweep_chart
from src.skab import SENSORS

pio.renderers.default = "notebook_connected"
RESULTS = ROOT / "results"
metrics = pd.read_csv(RESULTS / "metrics.csv")
sweep = pd.read_csv(RESULTS / "sweep.csv")
scores = pd.read_parquet(RESULTS / "scores.parquet")
meta = json.loads((RESULTS / "meta.json").read_text())
order = list(metrics["detector"].unique())

## Results

NAB windows are wide, so random alarms often land inside them. The chance F1 shifts each detector's test alarm sequence by a random
circular offset (1000 times), which keeps the number and length of alarm runs but breaks their alignment with the labels.
The p-value is the share of shifts reaching at least the observed F1.

Cells show event F1 (mean over seeds) with chance F1 in brackets; `*` marks p < 0.05.

In [2]:
per = metrics.groupby(["series", "detector"], sort=False).agg(
    f1=("f1", "mean"), chance=("chance_f1", "mean"), p=("p_value", "median"), events=("events", "first"),
)
cell = per.apply(lambda r: f"{r.f1:.2f} ({r.chance:.2f}){'*' if r.p < 0.05 else ''}", axis=1)
table = cell.unstack("detector")[order]
table.insert(0, "test events", per["events"].unstack("detector").iloc[:, 0])
table

detector,test events,zscore,iforest,matrix_profile,matrix_profile_raw,autoencoder,relative_entropy,ensemble
series,,,,,,,,
machine_temperature,2,0.20 (0.19),0.36 (0.20),0.00 (0.00),0.57 (0.22),0.48 (0.22),0.31 (0.23),0.50 (0.19)*
nyc_taxi,4,0.35 (0.28),0.39 (0.35),0.00 (0.08),0.44 (0.37),0.41 (0.33),0.73 (0.30),0.44 (0.33)
ambient_temperature,1,0.00 (0.13),0.00 (0.03),0.20 (0.15),0.33 (0.20),0.39 (0.17),0.22 (0.14),0.14 (0.10)
ec2_request_latency,2,0.50 (0.18),0.63 (0.18),0.36 (0.19),0.36 (0.20),0.83 (0.17)*,0.44 (0.19),0.40 (0.17)
cpu_asg_misconfig,1,0.05 (0.06),0.26 (0.13),1.00 (0.17),1.00 (0.23),0.47 (0.24),1.00 (0.15),0.10 (0.10)
rds_cpu,2,0.40 (0.29),0.42 (0.42),0.20 (0.33),0.44 (0.44),0.37 (0.38),0.67 (0.18),0.27 (0.23)
ec2_cpu,2,0.31 (0.26),0.29 (0.36),0.40 (0.26),0.80 (0.31),0.27 (0.36),0.00 (0.00),0.31 (0.26)
traffic_occupancy,2,0.67 (0.36),0.77 (0.34),0.33 (0.31),0.57 (0.35),0.42 (0.35),0.67 (0.20),0.50 (0.32)
traffic_speed,3,0.00 (0.00),0.86 (0.26),0.00 (0.00),0.86 (0.32)*,0.89 (0.31),0.80 (0.22),0.75 (0.28)


In [3]:
seed_sd = metrics.groupby(["series", "detector"])["f1"].std().groupby("detector").mean()
summary = per.groupby("detector").agg(
    f1=("f1", "mean"), chance=("chance", "mean"), significant=("p", lambda p: f"{(p < 0.05).sum()}/{len(p)}"),
)
summary["f1 − chance"] = summary["f1"] - summary["chance"]
summary["seed sd"] = seed_sd
summary = summary.join(metrics.groupby("detector")[["precision", "recall", "delay_h", "point_f1"]].mean())
summary.loc[order].round(2)

,f1,chance,significant,f1 − chance,seed sd,precision,recall,delay_h,point_f1
detector,,,,,,,,,
zscore,0.28,0.19,0/9,0.08,NaN,0.18,0.69,-3.51,0.02
iforest,0.44,0.25,0/9,0.19,0.05,0.33,0.80,-8.72,0.26
matrix_profile,0.28,0.17,0/9,0.11,NaN,0.22,0.56,-10.72,0.06
matrix_profile_raw,0.60,0.29,1/9,0.30,NaN,0.47,1.00,-6.44,0.40
autoencoder,0.50,0.28,1/9,0.22,0.07,0.37,0.94,-10.94,0.36
relative_entropy,0.54,0.18,0/9,0.36,NaN,0.57,0.74,-3.70,0.14
ensemble,0.38,0.22,1/9,0.16,NaN,0.25,1.00,-16.80,0.18


Delay is measured from the labeled anomaly point, which NAB places in the middle of each window, so values between minus and plus
half a window width are expected even for alarms without real lead time.

## Threshold sensitivity

Event F1 for a range of validation quantiles. A detector whose curve stays high over several quantiles ranks anomalies well;
a sharp peak means the result depends on the threshold choice. The best quantile is chosen on test data here and is therefore optimistic.

In [4]:
sweep_chart(sweep)

## Detections over time

Grey bands are the labeled anomaly windows, the dotted line marks the start of the test part.

In [5]:
fig = hero(scores, meta, "machine_temperature")
fig.write_image(ROOT / "docs" / "hero.png", width=1400, height=560, scale=2)
fig

In [6]:
for name in meta:
    small_multiples(scores, meta, name).show()

## Multivariate: SKAB

33 experiments on a water-circulation testbed with eight sensors sampled once per second (`other/2` is excluded because its anomaly
starts at point 104, before the training part ends). Following the SKAB convention, each experiment is fit on its first 300 points,
the threshold is the 99% quantile of points 300-400, and evaluation starts at point 400.

| Detector | Score | Contribution of sensor *i* |
|---|---|---|
| `zscore` | sum of squared standardized values | its own squared z-value (ignores correlations) |
| `pca` | combined T² + SPE index | complete decomposition contribution (Alcala & Qin 2011) |
| `autoencoder` | reconstruction error of a multichannel conv autoencoder | reconstruction error of channel *i* |

Before scoring, a causal moving average (EWMA, span 60 s) is subtracted from every sensor, so the detectors see deviations from the recent level instead of the slow drift of the testbed; the leave-one-out optimization below selected this for every detector. All scores are averaged over a trailing window of 20 s, and contributions always sum to the score.

Each experiment holds one long anomaly that covers about half of the test part. Two reference values help with reading the table:
- `chance_f1` comes from random time shifts of the alarms, pooled over all experiments.
- `point_f1 always-alarm` is the point-wise F1 of a detector that alarms on every test point. SKAB's own leaderboard reports point-wise F1.

`active_at_onset` is the share of experiments where the alarm was already on just before the anomaly started. In those cases the
delay of 0 s does not reflect real detection.

In [7]:
skab = pd.read_csv(RESULTS / "skab_metrics.csv")
skab_pooled = pd.read_csv(RESULTS / "skab_pooled.csv")
skab_meta = json.loads((RESULTS / "skab_meta.json").read_text())
summary = skab_pooled.groupby("detector")[["tp", "fp", "events", "f1", "chance_f1", "p_value"]].mean()
summary = summary.join(skab.groupby("detector").agg(
    point_f1=("point_f1", "mean"), delay_s=("delay_s", "median"), active_at_onset=("active_at_onset", "mean"),
))
summary["point_f1 always-alarm"] = np.mean([f["point_f1_always"] for f in skab_meta["files"].values()])
summary.round(2)

,tp,fp,events,f1,chance_f1,p_value,point_f1,delay_s,active_at_onset,point_f1 always-alarm
detector,,,,,,,,,,
autoencoder,32.0,79.0,33.0,0.44,0.44,0.47,0.48,6.0,0.30,0.7
pca,30.0,52.0,33.0,0.52,0.47,0.02,0.42,11.0,0.21,0.7
zscore,31.0,61.0,33.0,0.50,0.47,0.11,0.44,13.0,0.27,0.7


### Which sensor triggered the alarm?

The explanation for an event is the mean sensor contribution over all alarms inside the labeled window. It is checked in two ways.

**Injected faults.** On the anomaly-free SKAB recording, a step or a linear drift of 3σ or 6σ is added to one sensor for 200 s,
with the same local fit and threshold protocol as above. `top1_accuracy` is the share of detected faults where the top sensor is the
one that was shifted; guessing gives 1/8. `alarm_without_fault` is how often the same stretch alarms without any injection, which
shows how much of the detection is caused by drift in the data.

In [8]:
inj = pd.read_csv(RESULTS / "skab_injection.csv")
hit = inj[inj["detected"]]
keys = ["detector", "kind", "mag"]
inj.groupby(keys).agg(alarm_without_fault=("alarm_without_fault", "mean"), detected=("detected", "mean")).join(
    hit.groupby(keys).agg(top1_accuracy=("correct", "mean"))
).round(2)

alarm_without_fault  detected  top1_accuracy
detector    kind  mag                                              
autoencoder drift 3.0                  0.8      0.80           0.19
                  6.0                  0.8      0.82           0.30
            step  3.0                  0.8      0.92           0.65
                  6.0                  0.8      1.00           0.81
pca         drift 3.0                  0.9      0.90           0.17
                  6.0                  0.9      0.90           0.31
            step  3.0                  0.9      0.94           0.52
                  6.0                  0.9      0.94           0.85
zscore      drift 3.0                  0.8      0.79           0.11
                  6.0                  0.8      0.81           0.17
            step  3.0                  0.8      0.84           0.61
                  6.0                  0.8      0.91           0.81

In [9]:
hit.pivot_table(index="sensor", columns="detector", values="correct", aggfunc="mean").round(2)

detector,autoencoder,pca,zscore
sensor,,,
accel1,0.36,0.39,0.36
accel2,0.57,0.50,0.45
current,0.46,0.66,0.39
flow,0.59,0.33,0.44
fluid_temp,0.59,0.68,0.84
motor_temp,0.21,0.25,0.34
pressure,0.71,0.25,0.33
voltage,0.57,0.63,0.30


**Plausibility on real faults.** SKAB documents the fault type of each experiment, but not which sensor it affects. The expected
sensors below are a domain assumption: valve and leak experiments should show up in flow or pressure, rotor imbalance in the
accelerometers, hot water in the fluid temperature. Cavitation can affect hydraulics, vibration and motor current.

In [10]:
attr = pd.read_csv(RESULTS / "skab_attribution.csv")
attr.groupby(["cause", "detector"])["plausible"].mean().unstack().round(2)

detector,autoencoder,pca,zscore
cause,,,
draining until cavitation,1.00,1.00,1.00
fluid leak or addition,1.00,0.50,0.67
hot water supply,1.00,1.00,1.00
inlet valve closed,0.92,0.36,0.71
outlet valve closed,0.70,0.25,0.25
rotor imbalance,1.00,1.00,1.00
slow water increase,0.00,0.00,0.00
sudden water increase,1.00,0.00,1.00
two-phase inlet flow (cavitation),1.00,1.00,1.00


In [11]:
attribution_heatmap(attr[attr["seed"] == 0], "autoencoder", SENSORS)

## Streaming

The same detectors run point by point with a buffer of w + 1 samples (`src/streaming.py`). Because every score depends only on the
trailing window, the streamed scores match the batch scores up to float rounding; the tests check this for every detector.

![streaming detection](streaming.gif)

Top: anomaly score against the threshold. Middle: share of the score per sensor. Right: the current explanation.

In [12]:
pd.Series(json.loads((RESULTS / "streaming.json").read_text()))

file                             other/7
cause                    rotor imbalance
detector                     autoencoder
detrend                               60
w                                     20
q                                   0.99
threshold                      12.443395
points                              1090
ms_per_point                    0.665408
max_rel_diff_to_batch                0.0
dtype: object

## Optimization (leave-one-out)

Every setting is chosen on all *other* series and evaluated on the held-out one, so the numbers below are out-of-sample.
The grid covers the threshold quantile, EWMA smoothing of the score (0, ¼ or ½ of the window) and a minimum alarm duration
(0, 0.1 or ¼ of the window). Further options were added in stages:

- **A** post-processing only; SKAB uses the 60 s detrending already
- **B** ensembles over any subset of detectors (scores calibrated on normal validation data, then averaged)
- **C** context length: 0.5×, 1× or 2× the default for NAB, 10, 20 or 40 s for SKAB. Event merging and the post-window grace keep the default length, so the evaluation itself does not change
- **D** (SKAB only) fitting on the separate anomaly-free recording instead of the first 300 points of each experiment

Here the detector itself is part of the setting, which avoids picking the best detector after looking at the results.
`diff_vs_first` is the paired mean difference to stage A over series, with a bootstrap 95% interval.

In [13]:
pd.concat({ds: pd.read_csv(RESULTS / f"opt_{ds}_auto_summary.csv").set_index("stage") for ds in ("nab", "skab")}).round(3)

mean_f1  pooled_f1    tp    fp  diff_vs_first  \
     stage                                                                    
nab  A_previous                 0.536      0.571  16.0  21.0          0.000   
     B_ensemble_subsets         0.583      0.627  16.0  16.0          0.047   
     C_context_length           0.761      0.723  17.0  11.0          0.224   
skab A_previous                 0.601      0.556  30.0  45.0          0.000   
     B_ensemble_subsets         0.598      0.566  30.0  43.0         -0.003   
     C_context_length           0.540      0.504  29.0  53.0         -0.061   
     D_anomaly_free_training    0.540      0.504  29.0  53.0         -0.061   

                              ci_low  ci_high  \
     stage                                      
nab  A_previous                0.000    0.000   
     B_ensemble_subsets       -0.016    0.123   
     C_context_length          0.128    0.308   
skab A_previous                0.000    0.000   
     B_ensemble_subsets       -0.028    0.016   
     C_context_length         -0.109   -0.019   
     D_anomaly_free_training  -0.108   -0.020   

                                                             most_chosen  
     stage                                                                
nab  A_previous                                    matrix_profile_raw w1  
     B_ensemble_subsets       ensemble w1|autoencoder+matrix_profile_raw  
     C_context_length              ensemble w2|matrix_profile_raw+zscore  
skab A_previous                ensemble local-w20|autoencoder+pca+zscore  
     B_ensemble_subsets               ensemble local-w20|autoencoder+pca  
     C_context_length          ensemble local-w10|autoencoder+pca+zscore  
     D_anomaly_free_training   ensemble local-w10|autoencoder+pca+zscore

On NAB, adding the context length gives a clear and stable gain: nearly every fold picks the same ensemble. On SKAB none of
the stages beats A. Shorter windows look better on the full grid but do not carry over to held-out experiments, and models fit on
the anomaly-free recording do worse than local fits even with their best setting. The pipeline therefore uses stage C for NAB
and stage A for SKAB (`src/config.py`).

Per detector, for the selected stage (`default` is the untuned setting, chance F1 and p-value are pooled over all series):

In [14]:
from src.config import SELECTED_STAGE


def opt_table(ds):
    stage = SELECTED_STAGE[ds]
    loo = pd.read_csv(RESULTS / f"opt_{ds}_loo.csv").query("stage == @stage")
    pool = pd.read_csv(RESULTS / f"opt_{ds}_pooled.csv").query("stage == @stage").set_index(["detector", "setting"])
    t = loo.groupby(["detector", "setting"]).agg(
        mean_f1=("f1", "mean"), tp=("tp", "sum"), fp=("fp", "sum"), events=("events", "sum"),
    ).join(pool[["pooled_f1", "pooled_chance_f1", "pooled_p"]])
    return t.unstack("setting").round(3)


opt_table("nab")

mean_f1             tp            fp        events        \
setting            default  tuned default tuned default tuned default tuned   
detector                                                                      
autoencoder          0.577  0.523      18    13      32    13      19    19   
ensemble             0.330  0.761      16    17      65    11      19    19   
iforest              0.460  0.667      16    16      33    13      19    19   
matrix_profile       0.277  0.235       8     6      32    19      19    19   
matrix_profile_raw   0.598  0.586      19    18      34    24      19    19   
zscore               0.275  0.682      13    16      87    10      19    19   

                   pooled_f1        pooled_chance_f1        pooled_p         
setting              default  tuned          default  tuned  default  tuned  
detector                                                                     
autoencoder            0.522  0.578            0.315  0.253    0.001  0.001  
ensemble               0.320  0.723            0.247  0.203    0.017  0.001  
iforest                0.471  0.667            0.284  0.284    0.001  0.001  
matrix_profile         0.271  0.273            0.189  0.166    0.074  0.056  
matrix_profile_raw     0.528  0.590            0.302  0.304    0.001  0.001  
zscore                 0.218  0.711            0.176  0.247    0.101  0.001

In [15]:
opt_table("skab")

mean_f1             tp            fp        events        \
setting     default  tuned default tuned default tuned default tuned   
detector                                                               
autoencoder   0.520  0.596      32    32      75    54      33    33   
ensemble      0.599  0.601      31    30      53    45      33    33   
pca           0.569  0.581      30    30      52    45      33    33   
zscore        0.562  0.561      31    30      61    55      33    33   

            pooled_f1        pooled_chance_f1        pooled_p         
setting       default  tuned          default  tuned  default  tuned  
detector                                                              
autoencoder     0.457  0.538            0.439  0.483    0.170  0.004  
ensemble        0.530  0.556            0.484  0.486    0.025  0.009  
pca             0.522  0.556            0.474  0.481    0.023  0.003  
zscore          0.496  0.508            0.465  0.469    0.109  0.052

Most frequently chosen setting per detector across the leave-one-out folds of the selected stage:

In [16]:
def chosen(ds):
    loo = pd.read_csv(RESULTS / f"opt_{ds}_loo.csv")
    loo = loo[(loo["stage"] == SELECTED_STAGE[ds]) & (loo["setting"] == "tuned")]
    return loo.groupby("detector")[["variant", "q", "span", "k"]].agg(lambda c: c.value_counts().index[0])


pd.concat({ds: chosen(ds) for ds in ("nab", "skab")})

variant       q  span     k
     detector                                                                
nab  autoencoder                                       w1  0.9990  0.25  0.25
     ensemble                w2|matrix_profile_raw+zscore  0.9995  0.00  0.10
     iforest                                           w2  0.9995  0.25  0.10
     matrix_profile                                  w0.5  0.9950  0.50  0.25
     matrix_profile_raw                                w2  0.9990  0.00  0.25
     zscore                                            w2  0.9990  0.50  0.00
skab autoencoder                                local-w20  0.9950  0.00  0.25
     ensemble            local-w20|autoencoder+pca+zscore  0.9990  0.00  0.25
     pca                                        local-w20  0.9950  0.00  0.25
     zscore                                     local-w20  0.9800  0.00  0.25

## Holdout: fresh NAB series

Everything above was designed and tuned on the same nine series, so leave-one-out only partly protects against overfitting. The holdout
consists of **all 27 remaining real NAB series** that have at least one labeled window in the test part, chosen by that rule alone. Each
series uses a fixed context rule (max of 2 h and 12 samples, no weekly profile) and the post-processing that was frozen on the nine
tuning series. Nothing is refit on the holdout.

The published NAB detectors are scored on the same test segments from their result files. *published* uses the threshold Numenta
optimized on all 58 files (in-sample for the holdout); *frozen* picks the threshold on the nine tuning series only, the same footing as
ours. The NAB score follows `nab/sweeper.py` (verified: HTM reproduces 70.10 / 74.32 on the full corpus). With *points* every
alarm sample counts, as in NAB; *onsets* counts only the first sample of each alarm run.

In [17]:
hold = pd.read_csv(RESULTS / "holdout_summary.csv")
cols = ["tp", "fp", "events", "pooled_f1", "pooled_chance_f1", "nab_standard_onsets", "nab_standard_points"]
view = hold[hold["setting"] != "default"].pivot_table(index=["detector", "setting"], columns="group", values=["pooled_f1"])
view.columns = [c[1] for c in view.columns]
view = view[["tuning", "holdout"]].assign(drop=lambda d: d["holdout"] - d["tuning"])
view.sort_values("holdout", ascending=False).round(3)

tuning  holdout   drop
detector           setting                          
contextOSE         published   0.813    0.722 -0.090
                   frozen      0.824    0.634 -0.189
numenta            published   0.750    0.591 -0.159
relativeEntropy    published   0.800    0.557 -0.243
                   frozen      0.800    0.557 -0.243
htmjava            frozen      0.667    0.548 -0.119
ARTime             frozen      0.776    0.528 -0.247
                   published   0.776    0.528 -0.247
htmjava            published   0.630    0.490 -0.140
numenta            frozen      0.773    0.468 -0.304
autoencoder        frozen      0.711    0.394 -0.317
matrix_profile_raw frozen      0.679    0.388 -0.291
earthgeckoSkyline  published   0.605    0.385 -0.219
                   frozen      0.605    0.385 -0.219
knncad             frozen      0.407    0.366 -0.041
                   published   0.407    0.366 -0.041
randomCutForest    published   0.686    0.360 -0.326
twitterADVec       published   0.629    0.356 -0.273
                   frozen      0.629    0.356 -0.273
skyline            published   0.606    0.350 -0.256
iforest            frozen      0.667    0.346 -0.320
ensemble           frozen      0.756    0.315 -0.441
windowedGaussian   published   0.562    0.306 -0.257
randomCutForest    frozen      0.714    0.268 -0.446
zscore             frozen      0.566    0.259 -0.307
skyline            frozen      0.492    0.253 -0.238
windowedGaussian   frozen      0.517    0.211 -0.307
random             published   0.208    0.124 -0.084
matrix_profile     frozen      0.286    0.121 -0.164
random             frozen      0.236    0.091 -0.145

In [18]:
hold[(hold["group"] == "holdout") & (hold["setting"] != "default")].set_index(["detector", "setting"])[cols] \
    .sort_values("pooled_f1", ascending=False).round(3)

tp   fp  events  pooled_f1  pooled_chance_f1  \
detector           setting                                                   
contextOSE         published  26   15      31      0.722             0.137   
                   frozen     26   25      31      0.634             0.151   
numenta            published  26   31      31      0.591             0.141   
relativeEntropy    published  22   26      31      0.557             0.149   
                   frozen     22   26      31      0.557             0.149   
htmjava            frozen     23   30      31      0.548             0.151   
ARTime             published  28   47      31      0.528             0.157   
                   frozen     28   47      31      0.528             0.157   
htmjava            published  24   43      31      0.490             0.146   
numenta            frozen     26   54      31      0.468             0.150   
autoencoder        frozen     25   71      31      0.394             0.184   
matrix_profile_raw frozen     25   73      31      0.388             0.178   
earthgeckoSkyline  frozen     26   78      31      0.385             0.150   
                   published  26   78      31      0.385             0.150   
knncad             frozen     24   76      31      0.366             0.162   
                   published  24   76      31      0.366             0.162   
randomCutForest    published  27   92      31      0.360             0.128   
twitterADVec       published  24   80      31      0.356             0.142   
                   frozen     24   80      31      0.356             0.142   
skyline            published  21   68      31      0.350             0.126   
iforest            frozen     22   74      31      0.346             0.178   
ensemble           frozen     20   76      31      0.315             0.132   
windowedGaussian   published  22   91      31      0.306             0.124   
randomCutForest    frozen     28  150      31      0.268             0.123   
zscore             frozen     26  144      31      0.259             0.143   
skyline            frozen     28  162      31      0.253             0.125   
windowedGaussian   frozen     28  207      31      0.211             0.116   
random             published   7   75      31      0.124             0.145   
matrix_profile     frozen     15  201      31      0.121             0.122   
random             frozen     31  620      31      0.091             0.090   

                              nab_standard_onsets  nab_standard_points  
detector           setting                                              
contextOSE         published               72.517               72.517  
                   frozen                  70.972               70.617  
numenta            published               72.010               72.010  
relativeEntropy    published               58.361               58.006  
                   frozen                  58.361               58.006  
htmjava            frozen                  65.041               65.041  
ARTime             published               75.206               75.206  
                   frozen                  75.206               75.206  
htmjava            published               64.470               64.470  
numenta            frozen                  65.069               33.131  
autoencoder        frozen                  57.685             -690.804  
matrix_profile_raw frozen                  58.150             -668.387  
earthgeckoSkyline  frozen                  65.094               65.094  
                   published               65.094               65.094  
knncad             frozen                  59.231               59.231  
                   published               59.231               59.231  
randomCutForest    published               59.179               55.048  
twitterADVec       published               53.533               51.531  
                   frozen                  53.533               51.531  


The tuned ensemble does not transfer: pooled F1 falls from 0.76 on the tuning series to 0.32 on the holdout (chance 0.13). The single
autoencoder and the raw matrix profile hold up better (0.39), but on equal footing the online-learning detectors from NAB lead
(contextOSE 0.63, relativeEntropy 0.56, HTM 0.55). Our detectors find a similar share of events but raise two to three times as many
false alarms: they are fit once on the first 40 % and do not adapt when the level of a series drifts, which is common in the AWS and
Twitter series. The headline number from the optimization section is therefore an in-sample estimate for this corpus.

## Improving generalization

**Where the holdout false alarms come from** (`src/fp_analysis.py`). False alarm runs are spread evenly over the test part, so slow
drift after training is not the main cause. Most are periods of higher variance or patterns that stay inside the training range;
level shifts account for only 4–18 %. They concentrate on a few bursty series and on the short hourly AdExchange series, where the
validation part has only about 320 points and a 99.9 % quantile is close to the maximum.

In [19]:
fp = pd.read_csv(RESULTS / "fp_analysis.csv")
fp["kind"] = np.select([fp["level_shift_mad"] > 3, fp["spread_ratio"] > 2, fp["outside_train_range"]],
                       ["level shift", "variance change", "spike"], "in-range pattern")
pd.crosstab(fp["kind"], fp["detector"], normalize="columns").round(2)

detector,autoencoder,contextOSE,ensemble,matrix_profile_raw,zscore
kind,,,,,
in-range pattern,0.28,0.28,0.45,0.41,0.62
level shift,0.10,0.12,0.18,0.12,0.04
spike,0.07,0.20,0.04,0.07,0.06
variance change,0.55,0.40,0.33,0.40,0.28


**New protocol** (`src/cv.py`). With the holdout used once, all 36 series are evaluated by cross-validation over the five NAB
categories: every setting, the detector included, is chosen on the other four categories. Stages add one idea each to the
baseline A (quantile threshold, smoothing, minimum duration):

| Stage | Adds |
|---|---|
| B | refractory period: raw alarms closer than r·w are merged into one alert at its first point |
| C, D | threshold on a rolling median/MAD z-score of the detector score instead of a fixed validation quantile (±B) |
| E | B, C and D together |
| F | log transform for skewed non-negative series |
| G | context length and weekly profile estimated from the autocorrelation of the training part |
| H | everything |
| I | ensembles of diverse members (`matrix_profile_raw` + `autoencoder` + `relative_entropy`, all five) |

`relative_entropy` is a new detector (Wang et al. 2011, as in NAB) with equal-width bins over the training range plus two
overflow bins. Its binning was chosen on all 36 series, a small decision made on the evaluation data.

In [20]:
cv = pd.read_csv(RESULTS / "cv_summary_mean.csv")
cv[cv["scope"] == "auto"].set_index("stage")[["pooled_f1", "pooled_chance_f1", "tp", "fp", "events", "diff_vs_A", "ci_low", "ci_high"]].round(3)

,pooled_f1,pooled_chance_f1,tp,fp,events,diff_vs_A,ci_low,ci_high
stage,,,,,,,,
A_quantile,0.431,0.201,42,103,50,0.000,0.000,0.000
B_quantile_refractory,0.466,0.165,38,75,50,0.042,0.003,0.085
C_likelihood,0.435,0.187,40,94,50,-0.039,-0.176,0.089
D_likelihood_refractory,0.419,0.153,31,67,50,-0.048,-0.192,0.094
E_postprocessing,0.418,0.167,38,94,50,0.007,-0.060,0.069
F_log,0.406,0.198,39,103,50,-0.032,-0.103,0.019
G_acf_context,0.431,0.201,42,103,50,0.000,0.000,0.000
H_all,0.443,0.161,33,66,50,-0.012,-0.120,0.082
I_diverse_ensemble,0.466,0.165,38,75,50,0.042,0.002,0.085


In [21]:
cv.pivot_table(index="scope", columns="stage", values="pooled_f1").round(3)

stage,A_quantile,B_quantile_refractory,C_likelihood,D_likelihood_refractory,E_postprocessing,F_log,G_acf_context,H_all,I_diverse_ensemble
scope,,,,,,,,,
all5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.440,0.440
auto,0.431,0.466,0.435,0.419,0.418,0.406,0.431,0.443,0.466
autoencoder,0.469,0.507,0.396,0.407,0.507,0.442,0.434,0.488,0.507
ensemble,0.393,0.427,0.178,0.177,0.427,0.373,0.366,0.371,0.427
iforest,0.413,0.423,0.351,0.373,0.310,0.413,0.459,0.469,0.423
matrix_profile_raw,0.476,0.531,0.435,0.419,0.444,0.476,0.476,0.420,0.531
mp_ae_re,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.456,0.456
relative_entropy,0.386,0.481,0.422,0.409,0.481,0.411,0.386,0.444,0.481
zscore,0.345,0.352,0.104,0.234,0.352,0.322,0.320,0.411,0.352


Only the refractory period gives a gain whose bootstrap interval excludes zero (pooled F1 0.431 → 0.466, false alarms
103 → 75). The rolling threshold reduces false alarms for `zscore` and the ensemble but never wins the selection; log transform,
estimated context length and diverse ensembles do not help. Larger search spaces (E, H) score below B, the same overfitting as
before in smaller form. Selecting by median F1 or by the one-standard-error rule gives similar numbers without a secure
difference. Stage B is adopted, with its settings chosen on all 36 series (`results/final_config.json`); the app uses them.

## Final check: UCR Anomaly Archive

The frozen stage-B settings are applied unchanged to the 250 series of the UCR Anomaly Archive (Wu & Keogh 2021), which was not
used for any decision. Each series has one anomaly in the test part and no timestamps, so the context length is the dominant
autocorrelation period of the training part. *top1* is the archive's standard measure: the highest test score lies within
100 points of the anomaly. The z-normalized matrix profile (classic discord search) is included as a reference.

In [22]:
pd.read_csv(RESULTS / "ucr_summary.csv", index_col="detector").sort_values("top1", ascending=False).round(3)

,series,top1,chance_top1,tp,fp,mean_f1,pooled_f1
detector,,,,,,,
matrix_profile,250,0.456,0.024,110,1452,0.208,0.121
matrix_profile_raw,250,0.408,0.024,104,1629,0.190,0.105
autoencoder,250,0.388,0.024,56,340,0.190,0.173
relative_entropy,250,0.260,0.024,46,1266,0.082,0.059
zscore,250,0.204,0.024,24,533,0.070,0.059
iforest,250,0.192,0.024,59,941,0.141,0.094


The ranking from NAB carries over: `matrix_profile_raw` and the autoencoder locate 41 % and 39 % of the anomalies against
2.4 % by chance, close to the discord reference (46 %; the differences between the three are not significant). Event F1 with the
NAB thresholds is low, because a threshold calibrated on NAB's wide windows raises many alarms on long series with a single short
anomaly. Ranking transfers, the alarm threshold does not.

## SKAB leaderboard

The SKAB repository publishes the predictions of its leaderboard models. Re-scoring them with an independent implementation of the
SKAB metrics reproduces the leaderboard (Conv-AE F1 0.78, FAR 13.55 %, MAR 28.02 %, NAB 23.61, also missed changepoints and false
positives). Our detectors run on the same 34 files and test part (from point 400; fit on 0–300, threshold on 300–400). *tuned* uses the
post-processing chosen by leave-one-out on the other files.

In [23]:
lb = pd.read_csv(RESULTS / "skab_leaderboard.csv")
lb.set_index(["source", "detector", "setting"])[["f1", "far", "mar", "nab_standard", "nab_lowfp", "nab_lowfn", "missed_cps", "fps", "event_f1"]] \
    .sort_values("nab_standard", ascending=False).round(3)

f1      far      mar  \
source    detector         setting                                 
ours      autoencoder      tuned         0.470   20.743   63.809   
          ensemble         tuned         0.413   19.737   69.548   
          zscore           tuned         0.465   24.388   63.284   
          pca              tuned         0.447   21.732   65.821   
published MSCRED           as published  0.356   43.917   70.128   
ours      pca              default       0.470   23.572   63.073   
          zscore           default       0.484   26.120   60.919   
published T-squared+Q      as published  0.758   26.618   24.916   
          Conv-AE          as published  0.784   13.554   28.017   
          LSTM-AE          as published  0.741   29.964   25.918   
ours      autoencoder      default       0.513   26.355   57.693   
published T-squared        as published  0.660   19.211   42.597   
          MSET             as published  0.780   39.728   14.134   
          Vanilla AE       as published  0.391    2.593   75.155   
          Vanilla LSTM     as published  0.535   12.348   59.541   
          Isolation forest as published  0.287    2.557   82.891   
          ArimaFD          as published  0.000    0.009  100.000   
baseline  always alarm     -             0.698  100.000    0.000   

                                         nab_standard  nab_lowfp  nab_lowfn  \
source    detector         setting                                            
ours      autoencoder      tuned               46.034     38.228     51.424   
          ensemble         tuned               36.213     29.268     41.203   
          zscore           tuned               33.954     25.693     39.172   
          pca              tuned               32.741     25.256     37.575   
published MSCRED           as published        31.910     16.267     39.646   
ours      pca              default             28.253     16.188     34.583   
          zscore           default             27.052     13.308     34.308   
published T-squared+Q      as published        25.345     14.514     31.332   
          Conv-AE          as published        23.612     21.544     27.553   
          LSTM-AE          as published        23.511     20.106     25.910   
ours      autoencoder      default             21.366    -13.932     36.029   
published T-squared        as published        19.543     10.204     24.315   
          MSET             as published        13.843     10.221     17.365   
          Vanilla AE       as published        11.410      6.532     13.906   
          Vanilla LSTM     as published        11.308     -3.802     17.250   
          Isolation forest as published        10.738    -19.839     22.644   
          ArimaFD          as published        -0.087     -0.173     -0.058   
baseline  always alarm     -                   -1.472     -2.945     -0.982   

                                         missed_cps  fps  event_f1  
source    detector         setting                                  
ours      autoencoder      tuned                 48  159     0.541  
          ensemble         tuned                 62  140     0.554  
          zscore           tuned                 64  172     0.512  
          pca              tuned                 67  155     0.559  
published MSCRED           as published          57  342     0.472  
ours      pca              default               67  261     0.517  
          zscore           default               65  297     0.500  
published T-squared+Q      as published          72  232     0.633  
          Conv-AE          as published          82   23     0.689  
          LSTM-AE          as published          88   69     0.619  
ours      autoencoder      default               44  793     0.458  
published T-squared        as published          84  203     0.652  
          MSET             as published          96   66     0.629  
          Vanilla AE       as published         103  106     0.483 

Two readings. On the changepoint leaderboard (NAB score) the tuned autoencoder reaches 46.0, ahead of every published model
(best MSCRED 31.9). Detrending makes the detectors react to the onset of a fault, which is what this score rewards. On the point-wise
outlier leaderboard they reach only 0.41–0.51 F1, because the detrended score decays while a fault persists. That leaderboard is weak
evidence, though: an always-on alarm already scores 0.70 F1, more than T-squared and close to Conv-AE (0.78). Without detrending our
autoencoder also reaches 0.77, at a false alarm rate of 55 %.

A two-timescale score (mean of the calibrated detrended and raw scores) was tested with the same leave-one-out ablation as before. It
helps the autoencoder alone (pooled F1 0.54 → 0.59) but not the automatic selection (−0.03, 95 % CI −0.08 to +0.02), so it was not
adopted.

In [24]:
pd.read_csv(RESULTS / "opt_skab_dual_auto_summary.csv").set_index("stage").round(3)

,mean_f1,pooled_f1,tp,fp,diff_vs_first,ci_low,ci_high,most_chosen
stage,,,,,,,,
A_previous,0.601,0.556,30.0,45.0,0.000,0.000,0.000,ensemble local-w20|autoencoder+pca+zscore
B_ensemble_subsets,0.598,0.566,30.0,43.0,-0.003,-0.028,0.016,ensemble local-w20|autoencoder+pca
E_no_detrend,0.598,0.566,30.0,43.0,-0.003,-0.028,0.016,ensemble local-w20|autoencoder+pca
F_dual_timescale,0.572,0.541,30.0,48.0,-0.029,-0.082,0.022,autoencoder dual-w20
